# Multi-Output XGBoost

Code is based off [xgboostWB_electricity.py](https://github.com/Daniela-Shereen/GBRT-for-TSF/blob/main/XGBoost_(W-b)/Univariate/xgboostWB_electricity.py) example from research paper "Do We Really Need Deep Learning Models for Time Series Forecasting?" (Elsayed et al., 2021) with modifications for the research dataset.

## Set-Up

Load libraries and set variables that will be used in the modelling. Load the features data.

In [1]:
import numpy as np 
import math
import matplotlib.pyplot as plt
import pandas as pd 
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputRegressor
from pathlib import Path

In [2]:
VAL_YEAR, TEST_YEAR = 2018, 2019
NUM_PERIODS_INPUT = 48 # horizon to use as input
NUM_PERIODS_OUTPUT = 48 # horizon to predict
IN_DIR = Path("../data/NSW/processed")
OUT_DIR = Path("../data/NSW/processed/xgb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Load the enriched features dataset
features_parq = IN_DIR / "nsw_demand_features.parquet"
df = pd.read_parquet(features_parq)

## Pre-processing

Get a summary of the data as loaded in.

In [4]:
print(df.shape)
print(f"first observation: {min(df.index)}, last observation: {max(df.index)}")
print(df.columns.values)
df.describe()

(196513, 31)
first observation: 2010-01-01 00:00:00+00:00, last observation: 2021-03-18 00:00:00+00:00
['temp' 'y' 'y_forecast_latest' 'y_forecast_24h' 'hour' 'dow' 'is_weekend'
 'sin_hh' 'cos_hh' 'sin_dow' 'cos_dow' 'is_holiday' 'date'
 'is_long_weekend' 'is_hot_slot' 'is_cold_slot' 'is_hot_day' 'is_cold_day'
 'is_normal_day' 'y_lag1' 'y_lag48' 'y_lag336' 'y_roll_mean_48'
 'y_roll_std_48' 'y_roll_mean_336' 'temp_lag1' 'temp_lag2' 'temp_lag3'
 'temp_lag48' 'temp_roll_max_48' 'temp_roll_min_48']


,temp,y,y_forecast_latest,y_forecast_24h,hour,dow,is_weekend,sin_hh,cos_hh,sin_dow,...,y_lag336,y_roll_mean_48,y_roll_std_48,y_roll_mean_336,temp_lag1,temp_lag2,temp_lag3,temp_lag48,temp_roll_max_48,temp_roll_min_48
count,196513.000000,196513.000000,196513.000000,130999.000000,196513.00000,196513.000000,196513.000000,1.965130e+05,1.965130e+05,196513.000000,...,196177.000000,196489.000000,196489.000000,196417.000000,196512.000000,196511.000000,196510.000000,196465.000000,196489.000000,196489.000000
mean,17.526938,8113.145859,8122.251694,8163.879328,11.74994,3.000000,0.285783,-9.102662e-18,5.088722e-06,-0.000104,...,8114.295541,8113.214173,1001.296115,8113.703673,17.526933,17.526926,17.526918,17.526420,23.129053,12.602414
std,5.881148,1299.532774,1299.652719,1334.947598,6.92675,2.000244,0.451788,7.071068e-01,7.071104e-01,0.707161,...,1299.891392,794.857408,277.513273,653.871791,5.881163,5.881177,5.881191,5.881735,5.107634,5.358207
min,-1.300000,5074.630000,5115.770000,4831.720000,0.00000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.974928,...,5074.630000,5944.900625,353.881138,6578.555298,-1.300000,-1.300000,-1.300000,-1.300000,11.700000,-1.300000
25%,13.500000,7150.070000,7159.600000,7199.670000,5.50000,1.000000,0.000000,-7.071068e-01,-7.071068e-01,-0.781831,...,7150.920000,7554.742500,801.600415,7585.017560,13.500000,13.500000,13.500000,13.500000,19.150000,8.200000
50%,17.850000,8053.230000,8060.300000,8093.650000,11.50000,3.000000,0.000000,0.000000e+00,6.123234e-17,0.000000,...,8054.600000,8063.048958,971.987245,8080.426518,17.850000,17.850000,17.850000,17.850000,22.700000,12.800000
75%,21.500000,8958.550000,8969.020000,9012.120000,17.50000,5.000000,1.000000,7.071068e-01,7.071068e-01,0.781831,...,8960.210000,8643.396250,1153.577956,8558.597440,21.500000,21.500000,21.500000,21.500000,26.500000,17.100000
max,44.700000,14579.860000,14486.700000,14581.510000,23.50000,6.000000,1.000000,1.000000e+00,1.000000e+00,0.974928,...,14579.860000,11715.658125,2703.844474,10901.113512,44.700000,44.700000,44.700000,44.700000,44.700000,26.600000


In [5]:
# add year column which will be used to determine if training, val or test set
# remove columns which will not be used in the modelling
df = df.drop(columns = ['date', 'y_forecast_latest', 'y_forecast_24h',\
                        'is_hot_slot', 'is_cold_slot', 'is_hot_day', 'is_cold_day', 'is_normal_day',\
                        'temp'])
# check updated data
print(df.shape)
print(df.columns.values)
df.describe()

(196513, 22)
['y' 'hour' 'dow' 'is_weekend' 'sin_hh' 'cos_hh' 'sin_dow' 'cos_dow'
 'is_holiday' 'is_long_weekend' 'y_lag1' 'y_lag48' 'y_lag336'
 'y_roll_mean_48' 'y_roll_std_48' 'y_roll_mean_336' 'temp_lag1'
 'temp_lag2' 'temp_lag3' 'temp_lag48' 'temp_roll_max_48'
 'temp_roll_min_48']


,y,hour,dow,is_weekend,sin_hh,cos_hh,sin_dow,cos_dow,is_holiday,is_long_weekend,...,y_lag336,y_roll_mean_48,y_roll_std_48,y_roll_mean_336,temp_lag1,temp_lag2,temp_lag3,temp_lag48,temp_roll_max_48,temp_roll_min_48
count,196513.000000,196513.00000,196513.000000,196513.000000,1.965130e+05,1.965130e+05,196513.000000,196513.000000,196513.000000,196513.000000,...,196177.000000,196489.000000,196489.000000,196417.000000,196512.000000,196511.000000,196510.000000,196465.000000,196489.000000,196489.000000
mean,8113.145859,11.74994,3.000000,0.285783,-9.102662e-18,5.088722e-06,-0.000104,0.000215,0.033219,0.044455,...,8114.295541,8113.214173,1001.296115,8113.703673,17.526933,17.526926,17.526918,17.526420,23.129053,12.602414
std,1299.532774,6.92675,2.000244,0.451788,7.071068e-01,7.071104e-01,0.707161,0.707056,0.179209,0.206104,...,1299.891392,794.857408,277.513273,653.871791,5.881163,5.881177,5.881191,5.881735,5.107634,5.358207
min,5074.630000,0.00000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000,...,5074.630000,5944.900625,353.881138,6578.555298,-1.300000,-1.300000,-1.300000,-1.300000,11.700000,-1.300000
25%,7150.070000,5.50000,1.000000,0.000000,-7.071068e-01,-7.071068e-01,-0.781831,-0.900969,0.000000,0.000000,...,7150.920000,7554.742500,801.600415,7585.017560,13.500000,13.500000,13.500000,13.500000,19.150000,8.200000
50%,8053.230000,11.50000,3.000000,0.000000,0.000000e+00,6.123234e-17,0.000000,-0.222521,0.000000,0.000000,...,8054.600000,8063.048958,971.987245,8080.426518,17.850000,17.850000,17.850000,17.850000,22.700000,12.800000
75%,8958.550000,17.50000,5.000000,1.000000,7.071068e-01,7.071068e-01,0.781831,0.623490,0.000000,0.000000,...,8960.210000,8643.396250,1153.577956,8558.597440,21.500000,21.500000,21.500000,21.500000,26.500000,17.100000
max,14579.860000,23.50000,6.000000,1.000000,1.000000e+00,1.000000e+00,0.974928,1.000000,1.000000,1.000000,...,14579.860000,11715.658125,2703.844474,10901.113512,44.700000,44.700000,44.700000,44.700000,44.700000,26.600000


Following code is heavily based on [xgboostWB_electricity.py](https://github.com/Daniela-Shereen/GBRT-for-TSF/blob/main/XGBoost_(W-b)/Univariate/xgboostWB_electricity.py) from research paper "Do We Really Need Deep Learning Models for Time Series Forecasting?", adapted to the dataset for this project.

In [6]:
# apply min_max scaling to the lagged temp and demand features
# to be from -0.5 to 0.5 as mentioned in the paper
normalise_columns = ['y_lag1', 'y_lag48', 'y_lag336',\
                     'y_roll_mean_48', 'y_roll_std_48', 'y_roll_mean_336', \
                     'temp_lag1', 'temp_lag2', 'temp_lag3', 'temp_lag48', \
                     'temp_roll_max_48', 'temp_roll_min_48']

for colname in normalise_columns:
    df[colname] = MinMaxScaler(feature_range=(-0.5, 0.5)).fit_transform(np.array(df[colname]).reshape(-1,1))

df.describe()

,y,hour,dow,is_weekend,sin_hh,cos_hh,sin_dow,cos_dow,is_holiday,is_long_weekend,...,y_lag336,y_roll_mean_48,y_roll_std_48,y_roll_mean_336,temp_lag1,temp_lag2,temp_lag3,temp_lag48,temp_roll_max_48,temp_roll_min_48
count,196513.000000,196513.00000,196513.000000,196513.000000,1.965130e+05,1.965130e+05,196513.000000,196513.000000,196513.000000,196513.000000,...,196177.000000,196489.000000,196489.000000,196417.000000,196512.000000,196511.000000,196510.000000,196465.000000,196489.000000,196489.000000
mean,8113.145859,11.74994,3.000000,0.285783,-9.102662e-18,5.088722e-06,-0.000104,0.000215,0.033219,0.044455,...,-0.180211,-0.124258,-0.224500,-0.144852,-0.090719,-0.090719,-0.090719,-0.090730,-0.153665,-0.001706
std,1299.532774,6.92675,2.000244,0.451788,7.071068e-01,7.071104e-01,0.707161,0.707056,0.179209,0.206104,...,0.136755,0.137739,0.118093,0.151270,0.127851,0.127852,0.127852,0.127864,0.154777,0.192050
min,5074.630000,0.00000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000,...,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000
25%,7150.070000,5.50000,1.000000,0.000000,-7.071068e-01,-7.071068e-01,-0.781831,-0.900969,0.000000,0.000000,...,-0.281563,-0.221035,-0.309478,-0.267161,-0.178261,-0.178261,-0.178261,-0.178261,-0.274242,-0.159498
50%,8053.230000,11.50000,3.000000,0.000000,0.000000e+00,6.123234e-17,0.000000,-0.222521,0.000000,0.000000,...,-0.186492,-0.132951,-0.236972,-0.152550,-0.083696,-0.083696,-0.083696,-0.083696,-0.166667,0.005376
75%,8958.550000,17.50000,5.000000,1.000000,7.071068e-01,7.071068e-01,0.781831,0.623490,0.000000,0.000000,...,-0.091217,-0.032385,-0.159698,-0.041928,-0.004348,-0.004348,-0.004348,-0.004348,-0.051515,0.159498
max,14579.860000,23.50000,6.000000,1.000000,1.000000e+00,1.000000e+00,0.974928,1.000000,1.000000,1.000000,...,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000


In [7]:
# function used to generate window-based feature structure discussed in the paper
def window_processing(Data_df):
    Number_Of_Features = len(Data_df.columns)
    ############################################ Windowing ##################################
    end = len(Data_df)
    start = 0
    next = 0
    ts = [] # used to keep track of timestamps to allow for later train/val/test split
    x_batches = []
    y_batches = []  
    while (next+NUM_PERIODS_INPUT) < end:
        if start % 10000 == 0: print("row ", start) # status update
        next = start+NUM_PERIODS_INPUT
        ts.append(Data_df.index[next])
        x_batches.append(Data_df.iloc[start:next,:])
        y_batches.append(Data_df.iloc[next:next+NUM_PERIODS_OUTPUT,0]) # first col is y
        start = start+1
    # reformat x so that it is 3 dimensional (n observations, window size, features)
    x_batches=np.asarray(x_batches) 
    x_batches = x_batches.reshape(-1, NUM_PERIODS_INPUT, Number_Of_Features)
    # reformat y so that it is 3 dimensional (n observations, window size, 1)
    y_batches = np.asarray(y_batches)
    y_batches = y_batches.reshape(-1, NUM_PERIODS_OUTPUT, 1)     

    return x_batches, y_batches, ts

In [8]:
# run window-based preprocessing code
x_batches_Full = []
y_batches_Full = []
timestamps_Full = []
x_batches_Full, y_batches_Full, timestamps_Full = window_processing(df)
print("window-based preprocessing complete")

row  0
row  10000
row  20000
row  30000
row  40000
row  50000
row  60000
row  70000
row  80000
row  90000
row  100000
row  110000
row  120000
row  130000
row  140000
row  150000
row  160000
row  170000
row  180000
row  190000
window-based preprocessing complete


In [9]:
# check format of data
print(x_batches_Full.shape)
print(y_batches_Full.shape)
print(len(timestamps_Full))

(196418, 48, 22)
(196418, 48, 1)
196418


In [10]:
#=============== flatten each testing window into Instance =================================
all_instances_flattened = []
for i in range(0,len(x_batches_Full)):
    if i % 25000 == 0: print("updating item", i) # status update
    hold = []
    for j in range(0,len(x_batches_Full[i])):
        # take only the original features from the last timestamp
        if j == (len(x_batches_Full[i])-1):
            hold = np.concatenate((hold, x_batches_Full[i][j][:]), axis=None)
        else:
            hold = np.concatenate((hold, x_batches_Full[i][j][0]), axis=None)
    all_instances_flattened.append(hold)

print("flattening complete")

updating item 0
updating item 25000
updating item 50000
updating item 75000
updating item 100000
updating item 125000
updating item 150000
updating item 175000
flattening complete


In [11]:
# change structure of data so that it is 2D again
all_instances_x = np.reshape(all_instances_flattened, (len(x_batches_Full), -1))
all_instances_y = np.reshape(y_batches_Full, (len(y_batches_Full), NUM_PERIODS_OUTPUT))

In [12]:
# check resulting data
print(all_instances_x.shape)
print(all_instances_y.shape)

(196418, 69)
(196418, 48)


In [13]:
# clean feature and target training data
training_rows = [timestamps_Full[i].year<VAL_YEAR for i in range(len(timestamps_Full))]
Training_x = all_instances_x[training_rows,:]
Training_y = all_instances_y[training_rows,:]
# check format of resulting data
print(Training_x.shape)
print(Training_y.shape)
# last value in the demand window should match the first value in the test data
print('Check match: ', Training_x[1][NUM_PERIODS_INPUT-1], Training_y[0][0]) 

(140208, 69)
(140208, 48)
Check match:  7574.85 7574.85


In [14]:
# clean feature and target test data
test_rows = [timestamps_Full[i].year>=TEST_YEAR for i in range(len(timestamps_Full))]
Test_x = all_instances_x[test_rows,:]
Test_y = all_instances_y[test_rows,:]
# check format of resulting data
print(Test_x.shape)
print(Test_y.shape)
# last value in the demand window should match the first value in the test data
print('Check match: ', Test_x[1][NUM_PERIODS_INPUT-1], Test_y[0][0]) 

(38690, 69)
(38690, 48)
Check match:  7612.74 7612.74


## Train model

In [15]:
#=========================== CALLING XGBOOST ===========================
# model parameters as per the paper
# due to time constraints, there was no time to conduct own hyperparameter selection using cross-validation
model=xgb.XGBRegressor(learning_rate =0.2,
                       n_estimators=20,
                       max_depth=8,
                       min_child_weight=1,
                       gamma=0.0,
                       subsample=0.8,
                       colsample_bytree=0.8,
                       scale_pos_weight=1,
                       seed=42)

multioutput=MultiOutputRegressor(model).fit(Training_x,Training_y)

print('Fitting Done!')

Fitting Done!


## Predict on test data

In [16]:
#============================== PREDICTION ===============================
prediction=multioutput.predict(Test_x)
# check that shape of preduction matches actual
print('test ',Test_y.shape)
print('prediction ',prediction.shape)

test  (38690, 48)
prediction  (38690, 48)


In [17]:
# make it into a dataframe
OUT_PREFIX = "tplus"
prediction_df = pd.DataFrame(prediction, columns = [OUT_PREFIX+str(i) for i in range(NUM_PERIODS_OUTPUT)])
print(prediction_df.shape)
# add back in the timestamp
test_times = np.array(timestamps_Full)[test_rows]
prediction_df['timestamp'] = pd.Series(test_times)
print(prediction_df.shape)

(38690, 48)
(38690, 49)


In [18]:
# re-arrange data so that all the predictions for the same timestamp are gathered in the same row
AGG_PREFIX = "tminus"
for i in range(NUM_PERIODS_OUTPUT):
    prediction_df[AGG_PREFIX+str(i)] = prediction_df[OUT_PREFIX+str(i)].shift(i)

print(prediction_df.shape)
print(prediction_df.columns.values)

(38690, 97)
['tplus0' 'tplus1' 'tplus2' 'tplus3' 'tplus4' 'tplus5' 'tplus6' 'tplus7'
 'tplus8' 'tplus9' 'tplus10' 'tplus11' 'tplus12' 'tplus13' 'tplus14'
 'tplus15' 'tplus16' 'tplus17' 'tplus18' 'tplus19' 'tplus20' 'tplus21'
 'tplus22' 'tplus23' 'tplus24' 'tplus25' 'tplus26' 'tplus27' 'tplus28'
 'tplus29' 'tplus30' 'tplus31' 'tplus32' 'tplus33' 'tplus34' 'tplus35'
 'tplus36' 'tplus37' 'tplus38' 'tplus39' 'tplus40' 'tplus41' 'tplus42'
 'tplus43' 'tplus44' 'tplus45' 'tplus46' 'tplus47' 'timestamp' 'tminus0'
 'tminus1' 'tminus2' 'tminus3' 'tminus4' 'tminus5' 'tminus6' 'tminus7'
 'tminus8' 'tminus9' 'tminus10' 'tminus11' 'tminus12' 'tminus13'
 'tminus14' 'tminus15' 'tminus16' 'tminus17' 'tminus18' 'tminus19'
 'tminus20' 'tminus21' 'tminus22' 'tminus23' 'tminus24' 'tminus25'
 'tminus26' 'tminus27' 'tminus28' 'tminus29' 'tminus30' 'tminus31'
 'tminus32' 'tminus33' 'tminus34' 'tminus35' 'tminus36' 'tminus37'
 'tminus38' 'tminus39' 'tminus40' 'tminus41' 'tminus42' 'tminus43'
 'tminus44' 'tminu

In [19]:
# remove the outward forecasts for the timestamps
# keep just the forecasts for the given timestamp
drop_like = [OUT_PREFIX]
keep_cols = [c for c in prediction_df.columns if not any(k in c for k in drop_like)]
predicted_values_df = prediction_df[keep_cols]
print(predicted_values_df.shape)
print(predicted_values_df.columns.values)

(38690, 49)
['timestamp' 'tminus0' 'tminus1' 'tminus2' 'tminus3' 'tminus4' 'tminus5'
 'tminus6' 'tminus7' 'tminus8' 'tminus9' 'tminus10' 'tminus11' 'tminus12'
 'tminus13' 'tminus14' 'tminus15' 'tminus16' 'tminus17' 'tminus18'
 'tminus19' 'tminus20' 'tminus21' 'tminus22' 'tminus23' 'tminus24'
 'tminus25' 'tminus26' 'tminus27' 'tminus28' 'tminus29' 'tminus30'
 'tminus31' 'tminus32' 'tminus33' 'tminus34' 'tminus35' 'tminus36'
 'tminus37' 'tminus38' 'tminus39' 'tminus40' 'tminus41' 'tminus42'
 'tminus43' 'tminus44' 'tminus45' 'tminus46' 'tminus47']


In [20]:
# save the predicted data
predicted_values_df.to_csv(f"{OUT_DIR}/predicted_values_updated.csv")

In [21]:
# reformat from wide to long format
# each row is the prediction for a given timestamp
# slot is how far in advance it is being predicted
# xgb_pred is the predicted value
predicted_values_pivot = pd.wide_to_long(predicted_values_df, stubnames='tminus', i=['timestamp'], j='slot')
predicted_values_pivot.reset_index(inplace = True)
predicted_values_pivot.rename(columns={'tminus': 'xgb_pred'}, inplace=True)
predicted_values_pivot['timestamp'] = pd.to_datetime(predicted_values_pivot['timestamp'])
print(predicted_values_pivot.shape)
print(predicted_values_pivot.head())

(1857120, 3)
                  timestamp  slot     xgb_pred
0 2019-01-01 00:00:00+00:00     0  7511.634766
1 2019-01-01 00:30:00+00:00     0  7354.988281
2 2019-01-01 01:00:00+00:00     0  7260.702148
3 2019-01-01 01:30:00+00:00     0  7002.129395
4 2019-01-01 02:00:00+00:00     0  6742.121094


## Evaluation on test cohort

In [22]:
# Load in the test cohorts
COHORT_OVERALL_ALL  = f"{IN_DIR}/cohort3_overall_all.csv"
df_cohort_overall_all = pd.read_csv(COHORT_OVERALL_ALL)
print(df_cohort_overall_all.shape)

COHORT_HOTDAY_ALL  = f"{IN_DIR}/cohort3_hotday_all.csv"
df_cohort_hotday_all = pd.read_csv(COHORT_HOTDAY_ALL)
print(df_cohort_hotday_all.shape)

(38736, 6)
(3888, 6)


In [23]:
# combine predicted data with overall test cohort
df_cohort_overall_eval = df_cohort_overall_all.copy()
df_cohort_overall_eval['timestamp'] = pd.to_datetime(df_cohort_overall_eval['timestamp']).dt.tz_localize('UTC')
df_cohort_overall_eval = pd.merge(df_cohort_overall_eval, predicted_values_pivot, how='left')
# created filtered version to match midnight-to-midnight predictions for other models
overall_eval = df_cohort_overall_eval.copy()
overall_eval['timeslot'] = overall_eval['timestamp'].dt.hour * 2 + (overall_eval['timestamp'].dt.minute // 30)
overall_eval = overall_eval[overall_eval["slot"]==overall_eval["timeslot"]]

# combine predicted data with hot days test cohort
df_cohort_hotday_eval = df_cohort_hotday_all.copy()
df_cohort_hotday_eval['timestamp'] = pd.to_datetime(df_cohort_hotday_eval['timestamp']).dt.tz_localize('UTC')
df_cohort_hotday_eval = pd.merge(df_cohort_hotday_eval, predicted_values_pivot, how='left')
# created filtered version to match midnight-to-midnight predictions for other models
hotday_eval = df_cohort_hotday_eval.copy()
hotday_eval['timeslot'] = hotday_eval['timestamp'].dt.hour * 2 + (hotday_eval['timestamp'].dt.minute // 30)
hotday_eval = hotday_eval[hotday_eval['slot']==hotday_eval['timeslot']]

The following code draws from the evaluation format and code first provided by Maria Kim.

In [24]:
# ----- Metrics -----
def mae(y, yp):
    return np.nanmean(np.abs(yp - y))

def rmse(y, yp):
    e = yp - y
    return np.sqrt(np.nanmean(e * e))

def mape(y, yp):
    denom = np.where(y == 0, np.nan, np.abs(y))
    return np.nanmean(np.abs(yp - y) / denom) * 100

def bias(y, yp):
    return np.nanmean(yp - y)

def metric_table(y, yp):
    return pd.DataFrame([{
        "MAE_MW": mae(y, yp),
        "RMSE_MW": rmse(y, yp),
        "MAPE_%": mape(y, yp),
        "Bias_MW": bias(y, yp)
    }])

In [25]:
# ----- Loading & slot index -----
def load_cohort(df: pd.DataFrame, pred_col: str) -> pd.DataFrame:
    if pred_col not in df.columns:
        raise ValueError(f"Column '{pred_col}' not found in {csv.name}")
    
    return df[["timestamp", "slot", "timeslot", "true", pred_col]].rename(columns={pred_col: "yhat"})

In [26]:
# ----- By-slot aggregation -----
def by_hour_metrics(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for slot, g in df.groupby(group_col):
        y, yp = g["true"].to_numpy(), g["yhat"].to_numpy()
        rows.append({
            f"{group_col}": int(slot),
            "MAE_MW": mae(y, yp),
            "RMSE_MW": rmse(y, yp),
            "MAPE_%": mape(y, yp),
            "Bias_MW": bias(y, yp),
            "n": len(g)
        })
    return pd.DataFrame(rows).sort_values(group_col).reset_index(drop=True)

In [27]:
# ----- Plots -----
def plot_by_group(per_h: pd.DataFrame, group_col: str, title: str, out_png: Path):
    fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    s = per_h[group_col].to_numpy()
    axs = axs.ravel()

    axs[0].plot(s, per_h["MAE_MW"]);   axs[0].set_title("MAE by slot (MW)");   axs[0].set_ylabel("MAE")
    axs[1].plot(s, per_h["RMSE_MW"]);  axs[1].set_title("RMSE by slot (MW)")
    axs[2].plot(s, per_h["MAPE_%"]);   axs[2].set_title("MAPE by slot (%)");   axs[2].set_ylabel("MAPE (%)")
    axs[3].plot(s, per_h["Bias_MW"]);  axs[3].set_title("Bias by slot (pred − true, MW)"); axs[3].set_xlabel("Hour slot (0–47)")

    for ax in axs: ax.grid(True, alpha=0.3)
    fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180, bbox_inches="tight"); plt.close(fig)

def plot_by_group_compare(per_h_a: pd.DataFrame, label_a: str,
                          per_h_b: pd.DataFrame, label_b: str,
                          group_col: str, 
                          title: str, out_png: Path):

    # Expect columns: ["slot","MAE_MW","RMSE_MW","MAPE_%","Bias_MW"]
    fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axs = axs.ravel()
    s_a = per_h_a[group_col].to_numpy()
    s_b = per_h_b[group_col].to_numpy()

    def draw(ax, col, yl):
        ax.plot(s_a, per_h_a[col].to_numpy(), label=label_a, lw=1.6)
        ax.plot(s_b, per_h_b[col].to_numpy(), label=label_b, lw=1.6)
        ax.set_ylabel(yl); ax.grid(True, alpha=0.3); ax.legend()

    draw(axs[0], "MAE_MW",  "MAE (MW)")
    axs[0].set_title("MAE by Horizon (MW)")
    draw(axs[1], "RMSE_MW", "RMSE (MW)")
    axs[1].set_title("RMSE by Horizon (MW)")
    draw(axs[2], "MAPE_%",  "MAPE (%)")
    axs[2].set_title("MAPE by Horizon (%)")
    draw(axs[3], "Bias_MW", "Bias (MW)")
    axs[3].set_title("Bias by Horizon (pred − true, MW)")
    axs[3].set_xlabel("Horizon (slots ahead)")
    for ax in axs[:3]: ax.set_xlabel("")  # top row no x label

    fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close(fig)

def plot_error_hist(df: pd.DataFrame, title: str, out_png: Path):
    e = (df["yhat"] - df["true"]).to_numpy()
    plt.figure(figsize=(8,5))
    plt.hist(e, bins=60, alpha=0.85)
    plt.title(title); plt.xlabel("Signed error (MW)"); plt.ylabel("Count"); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(out_png, dpi=180, bbox_inches="tight"); plt.close()

In [28]:
# ----- One-shot evaluation for a cohort file -----
def evaluate_cohort(name: str, df: pd.DataFrame, pred_col: str = "xgb_pred", group_col: str = "slot"):
    print(f"[info] Evaluating {pred_col} on {name}")
    df = load_cohort(df, pred_col)

    # Overall metrics (all rows)
    overall_tbl = metric_table(df["true"].to_numpy(), df["yhat"].to_numpy())
    overall_tbl.index = [pred_col]
    overall_csv = OUT_DIR / f"{pred_col}_{name}_summary_final.csv"
    overall_tbl.to_csv(overall_csv)
    print(f"[save] summary -> {overall_csv}")

    # By-hour table & plots
    per_h = by_hour_metrics(df, group_col)
    per_h_csv = OUT_DIR / f"{pred_col}_{name}_by_{group_col}_final.csv"
    per_h.to_csv(per_h_csv, index=False)
    print(f"[save] by-hour table -> {per_h_csv}")

    curves_png = OUT_DIR / f"{pred_col}_{name}_by_{group_col}_curves_final.png"
    plot_by_group(per_h, group_col, f"{pred_col} — {name}", curves_png)
    print(f"[save] curves -> {curves_png}")

    # Error distribution (nice for “how skewed?” in intro)
    hist_png = OUT_DIR / f"{pred_col}_{name}_error_hist_final.png"
    plot_error_hist(df, f"{pred_col} — error distribution ({name})", hist_png)
    print(f"[save] error hist -> {hist_png}\n")

    return overall_tbl, per_h

In [29]:
overall_sum, overall_byh = evaluate_cohort("overall_all", overall_eval, pred_col="xgb_pred") 
hotday_sum,  hotday_byh  = evaluate_cohort("hotday_all",  hotday_eval,  pred_col="xgb_pred")

[info] Evaluating xgb_pred on overall_all
[save] summary -> ..\data\NSW\processed\xgb\xgb_pred_overall_all_summary_final.csv
[save] by-hour table -> ..\data\NSW\processed\xgb\xgb_pred_overall_all_by_slot_final.csv
[save] curves -> ..\data\NSW\processed\xgb\xgb_pred_overall_all_by_slot_curves_final.png
[save] error hist -> ..\data\NSW\processed\xgb\xgb_pred_overall_all_error_hist_final.png

[info] Evaluating xgb_pred on hotday_all
[save] summary -> ..\data\NSW\processed\xgb\xgb_pred_hotday_all_summary_final.csv
[save] by-hour table -> ..\data\NSW\processed\xgb\xgb_pred_hotday_all_by_slot_final.csv
[save] curves -> ..\data\NSW\processed\xgb\xgb_pred_hotday_all_by_slot_curves_final.png
[save] error hist -> ..\data\NSW\processed\xgb\xgb_pred_hotday_all_error_hist_final.png



In [30]:
# Combined 2-row summary
combined = pd.concat({
    "overall_all": overall_sum.iloc[0],
    "hotday_all":  hotday_sum.iloc[0],
}, axis=1).T.reset_index().rename(columns={"index": "cohort"})
combined_out = OUT_DIR / "xgboost_summary_both_final.csv"
combined.to_csv(combined_out, index=False)
print(f"[save] combined summary -> {combined_out}")

display(combined)

# Combined chart
compare_png = OUT_DIR / "xgboost_overall_vs_hotday_curves_final.png"
plot_by_group_compare(
    overall_byh, "overall_all",
    hotday_byh,  "hotday_all",
    group_col="slot",
    title="XGBoost forecast — overall vs hotday",
    out_png=compare_png,
)
print(f"[save] combined curves -> {compare_png}")

[save] combined summary -> ..\data\NSW\processed\xgb\xgboost_summary_both_final.csv


,cohort,MAE_MW,RMSE_MW,MAPE_%,Bias_MW
0,overall_all,335.062150,516.528594,4.235310,81.799086
1,hotday_all,547.628567,813.844059,5.769043,-332.377308


[save] combined curves -> ..\data\NSW\processed\xgb\xgboost_overall_vs_hotday_curves_final.png
